In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore, Pinecone
from langchain_classic.schema import Document
from pinecone import Pinecone, ServerlessSpec
import getpass
import os



In [22]:
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

In [24]:
# Initialization
index_name = "index"

if not pc.has_index(index_name):
    embedding_model = HuggingFaceEmbeddings()
    dimension = len(embedding_model.embed_query("dimension check"))

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

index = pc.Index(index_name)

In [25]:
doc1 = Document(
    page_content="Rohit Paudel is Nepal's current captain and most consistent batsman. Known for his elegant strokeplay and leadership, he has been crucial in Nepal's T20 World Cup qualifications.",
    metadata={"team": "Nepal National Team"}
)

doc2 = Document(
    page_content="Sandeep Lamichhane is Nepal's premier spinner and a T20 specialist. Famous for his mystery leg-spin and googly variations, he became the first Nepali to play in IPL for Delhi Daredevils.",
    metadata={"team": "Nepal National Team"}
)

doc3 = Document(
    page_content="Kushal Bhurtel is an explosive opener known for his aggressive batting. His quick starts and boundary-hitting ability make him Nepal's go-to powerplay batsman in T20Is.",
    metadata={"team": "Nepal National Team"}
)

doc4 = Document(
    page_content="Dipendra Singh Airee is a dynamic all-rounder who can bat aggressively and bowl handy off-spin. His match-winning performances, including a T20I six sixes in an over, highlight his potential.",
    metadata={"team": "Nepal National Team"}
)

doc5 = Document(
    page_content="Sompal Kami is Nepal's experienced fast bowler and lower-order hitter. Known for his pace and ability to swing the ball, he remains a key player in Nepal's limited-overs setup.",
    metadata={"team": "Nepal National Team"}
)

In [26]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [27]:
# recreate vector store with a proper embedding instance then add elements
# embedding = HuggingFaceEmbeddings()
vector_store = PineconeVectorStore(
    index_name='index',
    embedding=HuggingFaceEmbeddings()
)

In [28]:
vector_store.add_documents(docs)

['64f013fe-3cc6-4784-adba-049819c83933',
 'e44749ec-eaa4-4eca-9862-5bc1e8cfe656',
 '69994f2a-5c1f-4033-9b9d-d045229562dc',
 '6e2f410f-c716-4131-bc8a-3dff15c0066c',
 '53546239-5763-4cdb-81c0-76c00d4ca9eb']

In [30]:
# search documents
vector_store.similarity_search(
    query = 'Who is the best player?',
    k=2
)

[Document(id='64f013fe-3cc6-4784-adba-049819c83933', metadata={'team': 'Nepal National Team'}, page_content="Rohit Paudel is Nepal's current captain and most consistent batsman. Known for his elegant strokeplay and leadership, he has been crucial in Nepal's T20 World Cup qualifications."),
 Document(id='69994f2a-5c1f-4033-9b9d-d045229562dc', metadata={'team': 'Nepal National Team'}, page_content="Kushal Bhurtel is an explosive opener known for his aggressive batting. His quick starts and boundary-hitting ability make him Nepal's go-to powerplay batsman in T20Is.")]

In [32]:
vector_store.similarity_search_with_score(
    query='Who is the best player ?',
    k=1
)

[(Document(id='64f013fe-3cc6-4784-adba-049819c83933', metadata={'team': 'Nepal National Team'}, page_content="Rohit Paudel is Nepal's current captain and most consistent batsman. Known for his elegant strokeplay and leadership, he has been crucial in Nepal's T20 World Cup qualifications."),
  0.319946289)]

In [33]:
# delete items from vector store
vector_store.delete(ids='53546239-5763-4cdb-81c0-76c00d4ca9eb')

In [34]:
# query by turning into retriever
retriever = vector_store.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={"k":1, 'score_threshold': 0.4}
)

retriever.invoke("who is the best all rounder?")

[Document(id='6e2f410f-c716-4131-bc8a-3dff15c0066c', metadata={'team': 'Nepal National Team'}, page_content='Dipendra Singh Airee is a dynamic all-rounder who can bat aggressively and bowl handy off-spin. His match-winning performances, including a T20I six sixes in an over, highlight his potential.')]